# CT7 telemetry: CAN extraction

Foundations of Data Engineering and Analytics, data analysis project · Lucas Alois Griesser, Luca Jenewein, Nikola Adžić · Winter Semester 2026/27

Supporting notebook for the data analysis project *Forecasting the net battery energy of a Formula Student
electric car*. It decodes the raw CAN frames of the CT7 that are included in `data/raw/` and writes two files
to `data/`. Everything after this step, starting with `01_preprocessing.ipynb`, works only with these two files.

**Input (all part of the project package).**

| File | Content |
|---|---|
| `data/raw/S###.log.xz` | raw `candump` lines of the sessions with activity, one file per session: every thinned frame of the 10 selected messages inside an activity period, and the earliest thinned frame of these messages in the log file (time origin) |
| `data/ct7_signals.dbc` | definitions of these 10 messages, taken from CTM's DBC file `ctm.dbc` with `cantools` (IDs, bit layouts, scale, offset and unit unchanged) |
| `data/ct7_scan_stats.csv` | counts per log file from the one-time scan of the full logs |
| `data/ct7_archive_summary.csv` | which log files of the storage were selected |

The excerpt, the DBC excerpt and the two count files were written once by `make_raw_excerpt.py` from the full logs
on CTM's storage. That script is included only
to document where the excerpt comes from; it needs the original logs and DBC file and is not needed to run this
notebook. It applies the same file selection, thinning and activity rule as this notebook, so the values decoded here
are exactly those the full logs give. Counts that need the whole log file (all lines, malformed lines, frames with
another length, thinned frames outside activity periods, clock gaps) cannot be computed from the excerpt; they are read
from `ct7_scan_stats.csv`.

| Step | What is done |
|---|---|
| File selection | non-empty `candump-*.log` files named 26 May 2025 or later (scan); the earlier logs contain no driving with a battery power measurement |
| Signal selection | 12 signals from 10 CAN messages, each read from one fixed bus (ISA and AMS messages are mirrored on a second bus) |
| Decoding | CAN payloads to physical values with `cantools` and `data/ct7_signals.dbc` |
| Thinning | at most one frame per message and 100 ms; original timestamps are kept |
| Activity filter | only periods in which a wheel motor turns faster than 100 1/min, plus 30 s before and after |
| Pseudonymisation of the output | time in seconds since the earliest thinned frame of each log file, session IDs instead of file names, no dates (the file dates remain in `ct7_scan_stats.csv`) |

**Running.** Python 3.12 with the packages in `requirements.txt` (`pip install -r requirements.txt`). Run this
notebook first and then `01_preprocessing.ipynb`, both from the folder that contains them.

The raw excerpt keeps the original Unix timestamps of the frames; only the two output files are pseudonymised. Not
done here: resampling to a common time grid, range checks, handling of missing values, removal of repeated frames,
stint detection and features.

In [1]:
import csv
import gzip
import hashlib
import io
import logging
import lzma
from bisect import bisect_right
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import cantools
import pandas as pd

DATA = Path("data")
RAW = DATA / "raw"  # raw candump excerpt, one file per session
ACTIVE_RPM = 100    # a wheel motor faster than this counts as activity
PAD_S = 30          # seconds kept before and after an active period

## CAN messages and signals

`candump -l` writes one line per frame: `(unix time) bus ID#payload`. The car has four CAN buses
(`can-main`, `can-inv`, `can-sense`, `can-ams`). CTM's DBC file defines some frame IDs twice for different devices,
so each message is looked up by name and read from the bus on which that device sends it. `data/ct7_signals.dbc`
contains only the 10 messages used here.

In [2]:
logging.getLogger("cantools").setLevel(logging.ERROR)
db = cantools.database.load_file(DATA / "ct7_signals.dbc")
print(len(db.messages), "messages in the DBC excerpt")

SIGNALS = [  # (bus, DBC message, {DBC signal: column name})
    ("can-inv", "amk_fl_actual_values_1", {"actual_velocity": "rpm_fl"}),
    ("can-inv", "amk_fr_actual_values_1", {"actual_velocity": "rpm_fr"}),
    ("can-inv", "amk_rl_actual_values_1", {"actual_velocity": "rpm_rl"}),
    ("can-inv", "amk_rr_actual_values_1", {"actual_velocity": "rpm_rr"}),
    ("can-main", "pedals", {"accel": "accel_pct", "brake": "brake_pct", "regen": "regen_pct"}),
    ("can-main", "vcu_front_status", {"steering_angle": "steer_deg"}),
    ("can-main", "isa_current", {"value": "isa_current_a"}),
    ("can-main", "isa_ubat", {"value": "isa_voltage_v"}),
    ("can-main", "isa_power", {"value": "isa_power_w"}),
    ("can-main", "ams_ts_on_state", {"max_temp": "cell_temp_max_c"}),
]

WANTED = {}  # (bus, ID as written by candump) -> (message, columns)
rows = []
for bus, name, columns in SIGNALS:
    msg = db.get_message_by_name(name)
    WANTED[(bus, f"{msg.frame_id:03X}")] = (msg, columns)
    for sig, col in columns.items():
        s = msg.get_signal_by_name(sig)
        rows.append([col, f"{name}.{sig}", bus, f"0x{msg.frame_id:03X}", s.unit, s.scale, s.offset])
pd.DataFrame(rows, columns=["column", "DBC signal", "bus", "ID", "unit", "scale", "offset"]).set_index("column")

10 messages in the DBC excerpt


,DBC signal,bus,ID,unit,scale,offset
column,,,,,,
rpm_fl,amk_fl_actual_values_1.actual_velocity,can-inv,0x160,U/min,1.000000,0
rpm_fr,amk_fr_actual_values_1.actual_velocity,can-inv,0x161,U/min,1.000000,0
rpm_rl,amk_rl_actual_values_1.actual_velocity,can-inv,0x163,U/min,1.000000,0
rpm_rr,amk_rr_actual_values_1.actual_velocity,can-inv,0x162,U/min,1.000000,0
accel_pct,pedals.accel,can-main,0x100,%,1.000000,0
brake_pct,pedals.brake,can-main,0x100,%,1.000000,0
regen_pct,pedals.regen,can-main,0x100,%,1.000000,0
steer_deg,vcu_front_status.steering_angle,can-main,0x230,°,0.004578,0
isa_current_a,isa_current.value,can-main,0x1F0,A,0.001000,0


## Sessions and files

Each log file corresponds to one power-up of the car's logging computer (a Raspberry Pi) and is called a session. The
scan of the storage selected the files summarised below and numbered them S001, S002, ... in file-name order. The excerpt contains
the sessions in which a wheel motor turned.

In [3]:
archive = pd.read_csv(DATA / "ct7_archive_summary.csv").set_index("item").value
scan = pd.read_csv(DATA / "ct7_scan_stats.csv")
files = sorted(RAW.glob("S*.log.xz"))
print("scan of the full logs:")
print(archive.drop("bytes selected").to_string())
print(f"{archive['bytes selected'] / 2**30:.1f} GiB selected, {len(scan)} sessions")
print(f"excerpt: {len(files)} sessions with activity, {sum(p.stat().st_size for p in files) / 2**20:.1f} MiB compressed")

scan of the full logs:
item
log files                  528
empty files                 56
files before 2025-05-26    159
files selected             342
27.1 GiB selected, 342 sessions
excerpt: 120 sessions with activity, 36.5 MiB compressed


## Reading one session

The function reads the raw lines of a session and keeps the first frame of each wanted message in every 100 ms
interval. The excerpt already contains only such frames; applying the rule again changes nothing. Frames whose length
does not match the DBC (`dlc_mismatch`) are not in the excerpt and are counted by the scan: until 21 June 2025,
`vcu_front_status` was sent with 2 or 3 bytes and had no steering angle, so these frames cannot be decoded with the
current DBC. The scan also counts malformed lines (the last line of a file is often cut off) and two clock effects:

- **Forward gaps over 60 s.** The Raspberry Pi has no battery-backed clock. After booting it continues from an
  old time and jumps forward once it gets the time from the network (NTP).
- **Backward steps over 1 s.** These occur when one bus is written to the file later than the others.

Activity is determined from the four motor speeds. Periods closer than 60 s are merged. Times are converted to seconds
since the earliest frame in the excerpt of the session, which is the earliest thinned frame of the log file.

In [4]:
def active_periods(times):
    """Merge sorted activity timestamps into padded (start, end) periods."""
    periods = []
    for t in times:
        if periods and t - PAD_S <= periods[-1][1]:
            periods[-1][1] = t + PAD_S
        else:
            periods.append([t - PAD_S, t + PAD_S])
    return periods


def inside(t, periods, starts):
    i = bisect_right(starts, t) - 1
    return i >= 0 and t <= periods[i][1]


def read_session(path):
    frames, seen = [], set()
    with lzma.open(path, "rt") as f:
        for line in f:
            parts = line.split()
            t = float(parts[0][1:-1])
            can_id, payload = parts[2].split("#")
            key = (parts[1], can_id)
            if key not in WANTED or (key, int(t * 10)) in seen or len(payload) != 2 * WANTED[key][0].length:
                continue
            seen.add((key, int(t * 10)))
            frames.append((t, key, payload))
    t_first = min(t for t, _, _ in frames)

    decoded = []
    for t, key, payload in frames:
        msg, columns = WANTED[key]
        values = msg.decode(bytes.fromhex(payload), decode_choices=False)
        decoded += [(t, col, values[sig]) for sig, col in columns.items()]

    active = sorted(t for t, col, v in decoded if col.startswith("rpm_") and abs(v) > ACTIVE_RPM)
    periods = active_periods(active)
    starts = [a for a, _ in periods]
    keep = [(t, col, v) for t, col, v in decoded if inside(t, periods, starts)]
    stats = {"frames_in_excerpt": len(frames), "rows_written": len(keep),
             "active_s": round(sum(b - a for a, b in periods), 1)}
    day = datetime.fromtimestamp(active[0], ZoneInfo("Europe/Vienna")).date() if active else None
    return [(t - t_first, col, v) for t, col, v in keep], stats, day

## Extraction

The extract is a long table with one row per decoded value: `session_id, t_s, signal, value`. Rows are written in
file order, so `t_s` is not necessarily sorted.

In [5]:
sessions = []
with gzip.GzipFile(DATA / "ct7_can_extract.csv.gz", "wb", mtime=0) as gz, \
        io.TextIOWrapper(gz, newline="") as f:
    out = csv.writer(f)
    out.writerow(["session_id", "t_s", "signal", "value"])
    for sid in scan.session_id:
        path = RAW / f"{sid}.log.xz"
        if not path.exists():  # no activity in this log file
            continue
        values, stats, day = read_session(path)
        out.writerows([sid, f"{t:.6f}", col, f"{v:.6g}"] for t, col, v in values)
        sessions.append({"session_id": sid, "date": day, **stats})

sessions = scan.merge(pd.DataFrame(sessions), on="session_id", how="left").fillna(0)
print(len(sessions), "sessions,", int(sessions.rows_written.sum()), "values written;",
      "all excerpt lines used:", bool((sessions.frames_in_excerpt == sessions.excerpt_lines).all()))
print("from the scan:", int(sessions.lines.sum()), "lines,", int(sessions.malformed_lines.sum()), "malformed")
print("wanted frames with another length:", int(sessions.dlc_mismatch.sum()))

342 sessions, 7041760 values written; all excerpt lines used: True
from the scan: 603303358 lines, 219 malformed
wanted frames with another length: 2813154


## Session table

Days are numbered `D01`, `D02`, ... in date order, and only sessions with activity get a day. The date comes from the
first active timestamp. Because the clock may be wrong right after booting, it is compared with the date in the file
name. The dates are not written to `ct7_sessions.csv`. `frames_kept` (thinned frames of the whole file) and the line and clock counts come from
the scan; `rows_written` and `active_s` are computed above.

In [6]:
active = sessions[sessions.rows_written > 0].copy()
mismatch = (active.date.astype(str) != active.file_date).sum()
print(len(active), "sessions with activity; date differs from file name in", mismatch, "of them")
for _, r in active[active.date.astype(str) != active.file_date].iterrows():
    shift = (r.date - datetime.strptime(r.file_date, "%Y-%m-%d").date()).days
    print(f"  {r.session_id}: first activity {shift} day(s) after the date in the file name")

day_ids = {d: f"D{i:02d}" for i, d in enumerate(sorted(active.date.unique()), 1)}
sessions["day_id"] = sessions.date.map(day_ids)
columns = ["session_id", "day_id", "lines", "malformed_lines", "dlc_mismatch", "frames_kept", "rows_written", "active_s",
           "forward_gaps_over_60s", "backward_steps_over_1s"]
sessions = sessions[columns].astype({c: int for c in columns[2:] if c != "active_s"})
sessions.to_csv(DATA / "ct7_sessions.csv", index=False)
sessions[sessions.rows_written > 0].groupby("day_id").agg(
    sessions=("session_id", "size"), active_min=("active_s", lambda s: round(s.sum() / 60)),
    rows=("rows_written", "sum"))

120 sessions with activity; date differs from file name in 2 of them
  S036: first activity 1 day(s) after the date in the file name
  S041: first activity 4 day(s) after the date in the file name


,sessions,active_min,rows
day_id,,,
D01,5,31,164736
D02,2,3,5170
D03,15,100,628375
D04,1,1,7390
D05,5,15,87938
D06,3,27,178517
D07,1,7,44080
D08,2,2,8041
D09,1,1,6658


## Output check

The SHA-256 hashes identify the files; a rerun must give the same hashes.

In [7]:
extract = pd.read_csv(DATA / "ct7_can_extract.csv.gz")
for name in ["ct7_can_extract.csv.gz", "ct7_sessions.csv"]:
    path = DATA / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    print(f"{name}: {path.stat().st_size / 2**10:,.0f} KiB, sha256 {digest}")
print(extract.shape)
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(extract.groupby("signal").value.agg(["count", "min", "median", "max"]))

ct7_can_extract.csv.gz: 40,972 KiB, sha256 218cde592faf9989
ct7_sessions.csv: 12 KiB, sha256 8ee9414821094590
(7041760, 4)


,count,min,median,max
signal,,,,
accel_pct,612661,0.00,0.00,100.00
brake_pct,612661,0.00,0.00,100.00
cell_temp_max_c,607579,0.00,36.58,55.99
isa_current_a,608069,-173.45,0.00,"5,589.24"
isa_power_w,608007,"-28,508,200.00",-1.00,"932,873,000.00"
isa_voltage_v,608061,446.43,530.26,591.56
regen_pct,612661,0.00,0.00,100.00
rpm_fl,593458,"-1,670.00",0.00,"19,935.00"
rpm_fr,599701,"-1,608.00",0.00,"19,304.00"
